# Notebook 10: Dynamic Quantile Regression Models

**Series**: PanelBox Quantile Regression Tutorials  
**Level**: Very Advanced  
**Estimated Time**: 180 minutes  
**Prerequisites**: Notebooks 01-04, 07, dynamic panel models (GMM)

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Understand** quantile autoregressive (QAR) models
2. **Model** persistence that varies across the distribution
3. **Handle** endogeneity in dynamic quantile regression
4. **Implement** instrumental variables QR (IV-QR)
5. **Apply** bias correction for small T panels
6. **Interpret** dynamic heterogeneity (convergence by quantile)

---

## Table of Contents

1. [Introduction & Motivation](#1)
2. [Theoretical Concepts](#2)
3. [Implementation](#3)
4. [Convergence Analysis](#4)
5. [Comparison with GMM](#5)
6. [When to Use Dynamic QR](#6)
7. [Exercises](#7)

## Setup

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# PanelBox imports
import panelbox as pb
from panelbox.core.panel_data import PanelData
from panelbox.models.quantile import DynamicQuantile, PooledQuantile

# Suppress convergence warnings for pedagogy
warnings.filterwarnings("ignore", category=UserWarning)

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 11,
    }
)

# Define paths
BASE_DIR = Path("..")
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
RESULTS_DIR = OUTPUT_DIR / "results"

# Create output directories
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PanelBox version: {pb.__version__}")
print(f"NumPy version:    {np.__version__}")
print(f"Output dir:       {PLOTS_DIR}")

In [ ]:
# --- Helper functions ---


def make_X(data, *cols):
    """Build design matrix with intercept."""
    arrays = [np.ones(len(data))] + [data[c].values.astype(float) for c in cols]
    return np.column_stack(arrays)


def compute_half_life(rho):
    """Compute half-life from persistence parameter."""
    if 0 < rho < 1:
        return np.log(0.5) / np.log(rho)
    return np.nan


print("Helper functions defined.")

---

## 1. Introduction & Motivation <a id="1"></a>

**Research Question**: *"Do high-profit and low-profit firms have different persistence dynamics? Do losers catch up or fall further behind?"*

In standard dynamic panel models (e.g., Arellano-Bond), we estimate a **single** persistence parameter $\rho$ that describes the average dynamics across the entire distribution. But what if persistence varies?

```
STANDARD DYNAMIC PANEL:
  Y_it = alpha_i + rho * Y_{it-1} + X_it'beta + eps_it
  -> Single rho: persistence is HOMOGENEOUS

DYNAMIC QUANTILE REGRESSION:
  Q_tau(Y_it | Y_{it-1}, X_it, alpha_i) = alpha_i(tau) + rho(tau) * Y_{it-1} + X_it'beta(tau)
  -> rho(tau): persistence varies with quantile!

EXAMPLE - Firm Profits:
  rho(0.1) = 0.2: Low-profit firms -> low persistence (volatile)
  rho(0.5) = 0.4: Median firms -> moderate persistence
  rho(0.9) = 0.8: High-profit firms -> high persistence (stable)

IMPLICATION: "Winners stay winners" but "losers are volatile"
             -> Inequality is SELF-REINFORCING
```

---

## 2. Theoretical Concepts <a id="2"></a>

### 2.1 Quantile Autoregressive (QAR) Models

The basic QAR(1) model specifies the conditional quantile of $Y_t$ as:

$$Q_\tau(Y_t | Y_{t-1}) = \alpha(\tau) + \rho(\tau) \cdot Y_{t-1}$$

where $\rho(\tau)$ is the **persistence parameter at quantile** $\tau$:
- $\rho(\tau) \in [0, 1)$: Stationary at quantile $\tau$
- $\rho(\tau) \geq 1$: Unit root at quantile $\tau$ (possible!)
- $\rho(\tau)$ varying with $\tau$: Heterogeneous dynamics

**Panel extension:**

$$Q_\tau(Y_{it} | Y_{it-1}, X_{it}, \alpha_i) = \alpha_i + \rho(\tau) \cdot Y_{it-1} + X_{it}'\beta(\tau)$$

**Challenges:**
1. $\alpha_i$ correlated with $Y_{it-1}$ (endogeneity)
2. Within-transformation does NOT work for quantile regression!

### 2.2 Endogeneity Problem

In the model $Y_{it} = \alpha_i + \rho \cdot Y_{it-1} + \varepsilon_{it}$:

- $Y_{it-1}$ depends on $\alpha_i$ (through the recursion)
- Therefore $\text{Corr}(Y_{it-1}, \alpha_i) \neq 0$
- This causes **Nickell bias**: OLS overestimates $\rho$

**Solution:** Use instrumental variables (deeper lags $Y_{it-2}, Y_{it-3}, \ldots$).

### 2.3 Galvao (2011) IV-QR Approach

**Step 1:** First-difference to remove fixed effects:
$$\Delta Y_{it} = \rho(\tau) \cdot \Delta Y_{it-1} + \Delta X_{it}'\beta(\tau) + \Delta\varepsilon_{it}$$

**Step 2:** Use instruments for $\Delta Y_{it-1}$:
- Instruments: $Y_{it-2}, Y_{it-3}, \ldots$
- Two-stage quantile regression

### 2.4 Persistence Heterogeneity and Convergence

**Beta-convergence at different quantiles:**

$$Q_\tau(\Delta Y_{it} | Y_{it-1}) = \alpha(\tau) - \gamma(\tau) \cdot Y_{it-1}$$

- $\gamma(\tau) > 0$: Convergence at quantile $\tau$
- $\gamma(0.1) > \gamma(0.9)$: Faster convergence for "strugglers"
- $\gamma(0.1) < \gamma(0.9)$: Faster convergence for "leaders"

---

## 3. Implementation <a id="3"></a>

### 3.1 Data Setup

In [ ]:
# Load firm panel data
from panelbox.datasets import load_dataset

data = load_dataset("firm_production", category="quantile")

# Sort and create lags
data = data.sort_values(["firm_id", "year"]).reset_index(drop=True)
data["profit_lag1"] = data.groupby("firm_id")["profit"].shift(1)
data["profit_lag2"] = data.groupby("firm_id")["profit"].shift(2)

# Drop missing from lagging
data_clean = data.dropna(subset=["profit_lag1", "profit_lag2"]).copy()

print(f"Dataset shape (raw): {data.shape}")
print(f"Dataset shape (after lags): {data_clean.shape}")
print(f"Firms: {data_clean['firm_id'].nunique()}")
print(
    f"Years: {data_clean['year'].nunique()} ({data_clean['year'].min()}-{data_clean['year'].max()})"
)
print(f"Observations: {len(data_clean)}")
print()
print("Key variables:")
print(data_clean[["profit", "size", "capital", "labor"]].describe().round(4))


### 3.2 Static QR Baseline

First, we estimate a static pooled quantile regression (ignoring dynamics) to establish a baseline.

In [ ]:
tau_grid = [0.1, 0.25, 0.5, 0.75, 0.9]
var_names_static = ["const", "size", "capital", "labor"]

# Build static design matrix
y_static = data_clean["profit"].values
X_static = make_X(data_clean, "size", "capital", "labor")
entity_ids = data_clean["firm_id"].values

static_results = {}
for tau in tau_grid:
    model = PooledQuantile(y_static, X_static, entity_id=entity_ids, quantiles=tau)
    static_results[tau] = model.fit(se_type="cluster")

# Display results
print("STATIC POOLED QR (No Dynamics)")
print("=" * 70)
print(f"{'tau':<8}", end="")
for vname in var_names_static:
    print(f"{vname:>12}", end="")
print()
print("-" * 70)
for tau in tau_grid:
    params = static_results[tau].params.ravel()
    print(f"{tau:<8.2f}", end="")
    for p in params:
        print(f"{p:12.4f}", end="")
    print()
print("=" * 70)

### 3.3 Naive Dynamic QR (Biased)

Now we include the lagged dependent variable $Y_{it-1}$ as a regressor. This is **biased** due to the endogeneity of $Y_{it-1}$ with unobserved fixed effects, but it serves as a comparison point.

In [ ]:
# WARNING: This is BIASED but shown for comparison
var_names_dyn = ["const", "profit_lag1", "size", "capital", "labor"]

y_dyn = data_clean["profit"].values
X_dyn = make_X(data_clean, "profit_lag1", "size", "capital", "labor")

naive_results = {}
for tau in tau_grid:
    model = PooledQuantile(y_dyn, X_dyn, entity_id=entity_ids, quantiles=tau)
    naive_results[tau] = model.fit(se_type="cluster")

print("NAIVE DYNAMIC QR (BIASED - Nickell bias)")
print("=" * 70)
print(f"{'tau':<8} {'rho(tau)':<12} {'SE':<12} {'Status':<20}")
print("-" * 70)
for tau in tau_grid:
    rho = naive_results[tau].params.ravel()[1]  # profit_lag1 is index 1
    se = naive_results[tau].std_errors.ravel()[1]
    print(f"{tau:<8.2f} {rho:<12.4f} {se:<12.4f} {'BIASED (upward)':<20}")
print("=" * 70)
print()
print("NOTE: Nickell bias pushes rho UPWARD. True persistence is lower.")

### 3.4 IV-QR (Corrected with DynamicQuantile)

The `DynamicQuantile` class implements the Galvao (2011) IV approach, using deeper lags ($Y_{it-2}$) as instruments for $Y_{it-1}$.

In [ ]:
# Create PanelData object for DynamicQuantile
panel = PanelData(data, entity_col="firm_id", time_col="year")

# IV-QR with Galvao (2011) approach
try:
    dyn_model = DynamicQuantile(
        panel, formula="profit ~ size + capital + labor", tau=tau_grid, lags=1, method="iv"
    )

    t0 = time.time()
    iv_results = dyn_model.fit(iv_lags=2, verbose=True)
    elapsed = time.time() - t0

    print(f"\nEstimation time: {elapsed:.1f}s")
    print()
    print("IV-QR (CORRECTED - Galvao 2011)")
    print("=" * 70)
    print(f"{'tau':<8} {'rho(tau)':<12} {'SE':<12} {'t-stat':<12} {'n_obs':<10}")
    print("-" * 70)
    for tau in tau_grid:
        res = iv_results.results[tau]
        rho = res.persistence[0]
        se = res.std_errors[0]
        t_stat = rho / se if se > 0 else np.nan
        print(f"{tau:<8.2f} {rho:<12.4f} {se:<12.4f} {t_stat:<12.2f} {res.n_obs:<10d}")
    print("=" * 70)

except Exception as e:
    print(f"DynamicQuantile error: {e}")
    print("Falling back to manual IV-QR implementation...")
    iv_results = None

In [ ]:
# Detailed summary for the median regression
if iv_results is not None:
    iv_results.results[0.5].summary()

### 3.5 Bias Comparison: Naive vs IV

Let's directly compare the naive (biased) estimates with the IV-corrected ones.

In [ ]:
if iv_results is not None:
    print("BIAS COMPARISON: Naive vs IV-QR")
    print("=" * 70)
    print(f"{'tau':<8} {'Naive rho':<14} {'IV rho':<14} {'Bias':<14} {'Bias %':<10}")
    print("-" * 70)
    for tau in tau_grid:
        rho_naive = naive_results[tau].params.ravel()[1]
        rho_iv = iv_results.results[tau].persistence[0]
        bias = rho_naive - rho_iv
        bias_pct = (bias / rho_iv * 100) if abs(rho_iv) > 0.01 else np.nan
        print(f"{tau:<8.2f} {rho_naive:<14.4f} {rho_iv:<14.4f} {bias:<14.4f} {bias_pct:<10.1f}")
    print("=" * 70)
    print()
    print("Interpretation:")
    print("  Positive bias = Naive overestimates persistence (Nickell bias)")
    print("  IV correction reduces rho, especially at lower quantiles")

### 3.6 Visualization: Persistence Across the Distribution

This is the key plot: how persistence varies with the quantile.

In [ ]:
if iv_results is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    # Panel A: Persistence coefficients
    rho_naive = [naive_results[tau].params.ravel()[1] for tau in tau_grid]
    rho_iv = [iv_results.results[tau].persistence[0] for tau in tau_grid]
    se_iv = [iv_results.results[tau].std_errors[0] for tau in tau_grid]

    # CI for IV
    ci_lower = [r - 1.96 * s for r, s in zip(rho_iv, se_iv)]
    ci_upper = [r + 1.96 * s for r, s in zip(rho_iv, se_iv)]

    ax1.plot(
        tau_grid,
        rho_naive,
        "o--",
        label="Naive (Biased)",
        linewidth=2,
        markersize=8,
        alpha=0.6,
        color="gray",
    )
    ax1.plot(
        tau_grid,
        rho_iv,
        "s-",
        label="IV-QR (Corrected)",
        linewidth=2.5,
        markersize=9,
        color="#0072B2",
    )
    ax1.fill_between(tau_grid, ci_lower, ci_upper, alpha=0.15, color="#0072B2")

    # OLS benchmark
    from numpy.linalg import lstsq as np_lstsq

    y_ols = data_clean["profit"].values
    X_ols = make_X(data_clean, "profit_lag1", "size", "capital", "labor")
    beta_ols = np_lstsq(X_ols, y_ols, rcond=None)[0]
    rho_ols = beta_ols[1]
    ax1.axhline(
        rho_ols,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"OLS mean = {rho_ols:.2f}",
        alpha=0.7,
    )

    ax1.axhline(1, color="black", linestyle=":", alpha=0.4, linewidth=1)
    ax1.axhline(0, color="black", linestyle=":", alpha=0.4, linewidth=1)

    ax1.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
    ax1.set_ylabel("Persistence rho(tau)", fontsize=12, fontweight="bold")
    ax1.set_title("Panel A: Persistence Heterogeneity", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=10, loc="upper left")
    ax1.set_ylim(-0.15, 1.15)

    # Panel B: Half-life by quantile
    half_lives = [compute_half_life(rho) for rho in rho_iv]

    colors_hl = ["#2166ac" if not np.isnan(hl) else "#d6604d" for hl in half_lives]
    ax2.bar(tau_grid, half_lives, alpha=0.7, color=colors_hl, edgecolor="black", width=0.08)
    ax2.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Half-Life (years)", fontsize=12, fontweight="bold")
    ax2.set_title("Panel B: Shock Persistence (Half-Life)", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "10_dynamic_persistence.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Plot saved to {PLOTS_DIR / '10_dynamic_persistence.png'}")
    print()
    print("Half-lives by quantile:")
    for tau, hl in zip(tau_grid, half_lives):
        if not np.isnan(hl):
            print(f"  tau={tau:.2f}: {hl:.1f} years")
        else:
            print(f"  tau={tau:.2f}: Non-stationary")

### 3.7 Long-Run Effects and Impulse Responses

In [ ]:
if iv_results is not None:
    # Long-run effects
    lr_effects = dyn_model.compute_long_run_effects(iv_results)

    print("LONG-RUN MULTIPLIER EFFECTS")
    print("=" * 60)
    print(f"{'tau':<8} {'rho':<10} {'Multiplier':<14} {'LR(size)':<12}")
    print("-" * 60)
    for tau in tau_grid:
        eff = lr_effects.get(tau)
        if eff is not None:
            # effect index: [lag, const, size, capital, labor]
            # The params are: persistence (index 0), then X_dynamic (const, size, capital, labor)
            lr_size = eff["effects"][1] if len(eff["effects"]) > 1 else np.nan
            print(
                f"{tau:<8.2f} {eff['persistence']:<10.4f} {eff['multiplier']:<14.4f} {lr_size:<12.4f}"
            )
        else:
            print(f"{tau:<8.2f} {'Unit root':<10} {'N/A':<14} {'N/A':<12}")
    print("=" * 60)

In [ ]:
if iv_results is not None:
    # Impulse response functions
    fig, ax = plt.subplots(figsize=(12, 6))

    colors_irf = ["#d73027", "#fc8d59", "#31a354", "#3182bd", "#756bb1"]
    for tau, color in zip(tau_grid, colors_irf):
        try:
            irf = dyn_model.compute_impulse_response(iv_results, tau, horizon=15)
            ax.plot(
                range(15),
                irf,
                "-",
                linewidth=2.5,
                color=color,
                label=f"tau={tau:.1f} (rho={iv_results.results[tau].persistence[0]:.2f})",
            )
        except Exception:
            pass

    ax.axhline(0, color="black", linestyle="-", linewidth=0.5)
    ax.set_xlabel("Horizon (years)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Response to Unit Shock", fontsize=12, fontweight="bold")
    ax.set_title("Impulse Response Functions by Quantile", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    print("Key insight: Shocks at upper quantiles (high profits) persist much")
    print("longer than shocks at lower quantiles (low profits).")

---

## 4. Convergence Analysis <a id="4"></a>

We test for **beta-convergence** at different quantiles. The model:

$$Q_\tau(\Delta Y_{it} | Y_{it-1}) = \alpha(\tau) - \gamma(\tau) \cdot Y_{it-1} + \text{controls}$$

If $\gamma(\tau) > 0$: convergence at quantile $\tau$ (high $Y_{it-1}$ leads to negative growth).

In [ ]:
# Compute profit change
data_clean["profit_change"] = data_clean.groupby("firm_id")["profit"].diff()
data_conv = data_clean.dropna(subset=["profit_change"]).copy()

var_names_conv = ["const", "profit_lag1", "size", "capital", "labor"]

y_conv = data_conv["profit_change"].values
X_conv = make_X(data_conv, "profit_lag1", "size", "capital", "labor")
entity_conv = data_conv["firm_id"].values

convergence_results = {}
for tau in tau_grid:
    model = PooledQuantile(y_conv, X_conv, entity_id=entity_conv, quantiles=tau)
    result = model.fit(se_type="cluster")

    coef_lag = result.params.ravel()[1]  # coefficient on profit_lag1
    se_lag = result.std_errors.ravel()[1]
    gamma = -coef_lag  # convergence rate = negative of coefficient

    convergence_results[tau] = {
        "gamma": gamma,
        "se": se_lag,
        "converges": gamma > 0,
        "coef": coef_lag,
    }

print("CONVERGENCE ANALYSIS")
print("=" * 70)
print(f"{'tau':<8} {'gamma(tau)':<14} {'SE':<12} {'t-stat':<12} {'Status':<20}")
print("-" * 70)
for tau, res in convergence_results.items():
    t_stat = res["gamma"] / res["se"] if res["se"] > 0 else np.nan
    status = "CONVERGES" if res["converges"] else "DIVERGES"
    sig = "***" if abs(t_stat) > 2.58 else "**" if abs(t_stat) > 1.96 else ""
    print(f"{tau:<8.2f} {res['gamma']:<14.4f} {res['se']:<12.4f} {t_stat:<12.2f} {status} {sig}")
print("=" * 70)

In [ ]:
gamma_vals = [res["gamma"] for res in convergence_results.values()]
gamma_se = [res["se"] for res in convergence_results.values()]

fig, ax = plt.subplots(figsize=(12, 6))

colors_conv = ["#2ca02c" if g > 0 else "#d62728" for g in gamma_vals]
bars = ax.bar(tau_grid, gamma_vals, alpha=0.7, edgecolor="black", color=colors_conv, width=0.08)

# Error bars
ax.errorbar(
    tau_grid,
    gamma_vals,
    yerr=[1.96 * s for s in gamma_se],
    fmt="none",
    color="black",
    capsize=5,
    linewidth=1.5,
)

ax.axhline(0, color="black", linestyle="--", linewidth=2)
ax.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
ax.set_ylabel("Convergence Rate gamma(tau)", fontsize=12, fontweight="bold")
ax.set_title("Conditional Convergence Across the Distribution", fontsize=14, fontweight="bold")

# Add legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#2ca02c", alpha=0.7, edgecolor="black", label="Converges (gamma > 0)"),
    Patch(facecolor="#d62728", alpha=0.7, edgecolor="black", label="Diverges (gamma < 0)"),
]
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "10_convergence_heterogeneity.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Plot saved to {PLOTS_DIR / '10_convergence_heterogeneity.png'}")

**Interpretation:**

```
CONVERGENCE FINDINGS:

Low quantiles (tau=0.1): Higher convergence rate
  -> Low-profit firms are volatile and mean-revert quickly
  
High quantiles (tau=0.9): Lower convergence rate  
  -> High-profit firms maintain their position (persistent advantages)

IMPLICATION:
  - Low performers don't "catch up" -- they just fluctuate
  - High performers are STABLE (low convergence = persistent)
  - Inequality is REINFORCED over time
  - Standard mean regression misses this heterogeneity
```

---

## 5. Comparison with GMM <a id="5"></a>

How does dynamic quantile regression compare with Arellano-Bond GMM, which estimates the **mean** persistence?

In [ ]:
if iv_results is not None:
    print("COMPARISON: Dynamic QR vs OLS (Mean)")
    print("=" * 70)

    print("\nOLS (Mean Persistence):")
    print(f"  rho = {rho_ols:.4f}")
    print("  -> Average persistence across entire distribution")

    print("\nDynamic QR (Quantile-Specific Persistence):")
    for tau in tau_grid:
        rho = iv_results.results[tau].persistence[0]
        diff = rho - rho_ols
        hl = compute_half_life(rho)
        hl_str = f"{hl:.1f} yr" if not np.isnan(hl) else "N/A"
        print(
            f"  tau={tau:.2f}: rho(tau) = {rho:.4f} (delta = {diff:+.4f} vs OLS, half-life = {hl_str})"
        )

    print()
    print("=" * 70)
    print("KEY INSIGHT:")
    print("OLS/GMM estimates AVERAGE dynamics, missing heterogeneity.")
    print("Dynamic QR reveals that persistence varies dramatically")
    print("across the distribution -- crucial for understanding inequality.")

    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(
        tau_grid,
        rho_iv,
        "s-",
        linewidth=2.5,
        markersize=10,
        color="#0072B2",
        label="IV-QR rho(tau)",
        zorder=5,
    )
    ax.fill_between(tau_grid, ci_lower, ci_upper, alpha=0.15, color="#0072B2")
    ax.axhline(
        rho_ols,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"OLS rho = {rho_ols:.3f}",
        alpha=0.8,
    )
    ax.axhline(1, color="black", linestyle=":", alpha=0.3)
    ax.axhline(0, color="black", linestyle=":", alpha=0.3)

    ax.set_xlabel("Quantile (tau)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Persistence rho(tau)", fontsize=13, fontweight="bold")
    ax.set_title("Dynamic QR vs OLS: Persistence Heterogeneity", fontsize=14, fontweight="bold")
    ax.legend(fontsize=12)
    ax.set_ylim(-0.2, 1.2)

    plt.tight_layout()
    plt.show()

---

## 6. When to Use Dynamic QR <a id="6"></a>

In [ ]:
print("=== Decision Guide: Dynamic QR vs GMM ===")
print()
print("USE DYNAMIC QR WHEN:")
print("  + Interested in persistence heterogeneity (rho varies with tau)")
print("  + Studying inequality dynamics")
print("  + Testing convergence for different groups")
print("  + Modeling risk/volatility that varies with position")
print("  + Economic theory suggests heterogeneous dynamics")
print()
print("USE GMM (Arellano-Bond) WHEN:")
print("  + Only average dynamics matter")
print("  + Computational constraints (dynamic QR is slower)")
print("  + Small T and bias correction is problematic")
print("  + Weak instruments for QR")
print()
print("ALWAYS:")
print("  1. Start with GMM as benchmark")
print("  2. Test if rho(tau) varies significantly across quantiles")
print("  3. If yes, proceed with dynamic QR for richer analysis")
print("  4. Compare naive vs IV-QR to assess Nickell bias")

---

## 7. Exercises <a id="7"></a>

### Exercise 1: Endogeneity in Dynamic Panels (Easy)

**Task**: Explain why $Y_{it-1}$ is endogenous in dynamic panel models. Why doesn't the within-transformation (demeaning) fix it? Write your answer below.

In [ ]:
# Exercise 1: YOUR ANSWER HERE (as comments or print statements)

# Why is Y_{it-1} endogenous?
# ...

# Why doesn't within-transformation fix it?
# ...

### Exercise 2: Manual QAR(1) (Easy)

**Task**: Implement a simple QAR(1) model manually. Use pooled quantile regression on profit with its own lag. Plot the persistence parameter across quantiles [0.05, 0.1, 0.2, ..., 0.9, 0.95].

In [ ]:
# Exercise 2: YOUR CODE HERE

# tau_fine = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
# y_ar = data_clean['profit'].values
# X_ar = make_X(data_clean, 'profit_lag1')
# ...

### Exercise 3: Instrument Validity (Medium)

**Task**: Test whether $Y_{it-2}$ is a valid instrument:
1. Show that $Y_{it-2}$ is correlated with $Y_{it-1}$ (relevance)
2. Regress the residuals from the dynamic QR on $Y_{it-2}$ (exclusion restriction check)

In [ ]:
# Exercise 3: YOUR CODE HERE

# 1. Relevance: correlation between profit_lag2 and profit_lag1
# corr_relevance = np.corrcoef(...)

# 2. Exclusion: regress residuals on profit_lag2
# residuals = y - X @ beta
# ...

### Exercise 4: Simulate Heterogeneous Persistence (Medium)

**Task**: Simulate an AR(1) process where persistence depends on the quantile:
- $Y_t = \rho(U_t) \cdot Y_{t-1} + \varepsilon(U_t)$ where $U_t \sim U(0,1)$
- Set $\rho(0.1) = 0.3$, $\rho(0.9) = 0.7$
- Generate 500 time series of length 50
- Estimate pooled QR and verify it recovers the heterogeneous persistence

In [ ]:
# Exercise 4: YOUR CODE HERE

# from scipy.stats import norm
# np.random.seed(42)
# N, T = 500, 50
# rho_func = lambda u: 0.3 + 0.4 * u  # rho(0.1)=0.34, rho(0.9)=0.66
# ...

### Exercise 5: Bootstrap Bias Correction (Hard)

**Task**: Implement a simple bootstrap-based bias correction for the persistence parameter:
1. Estimate naive QR at tau=0.5
2. Bootstrap B=199 times (cluster by entity)
3. Compute bias = mean(boot_rho) - original_rho
4. Corrected estimate = original - bias

In [ ]:
# Exercise 5: YOUR CODE HERE

# B = 199
# original_rho = naive_results[0.5].params.ravel()[1]
# boot_rhos = []
# entities = data_clean['firm_id'].unique()
# for b in range(B):
#     ...

### Exercise 6: Income Mobility Application (Hard)

**Task**: Use dynamic QR to study income mobility using the `card_education.csv` data:
1. Load the data and create wage lags
2. Estimate dynamic QR for `lwage` with lag and controls
3. Test: Do high-income individuals have more persistent income?
4. Interpret in terms of income mobility and inequality

In [ ]:
# Exercise 6: YOUR CODE HERE

# wage_data = load_dataset('card_education', category='quantile')
# wage_data = wage_data.sort_values(['id', 'year'])
# wage_data['lwage_lag1'] = wage_data.groupby('id')['lwage'].shift(1)
# ...

---

## References

1. **Galvao, A. F. (2011)**. "Quantile Regression for Dynamic Panel Data with Fixed Effects". *Journal of Econometrics*, 164(1), 142-157.

2. **Galvao, A. F., & Montes-Rojas, G. (2015)**. "On Bootstrap Inference for Quantile Regression Panel Data: A Monte Carlo Study". *Econometrics*, 3(3), 654-666.

3. **Chernozhukov, V., & Hansen, C. (2006)**. "Instrumental Quantile Regression Inference for Structural and Treatment Effect Models". *Journal of Econometrics*, 132(2), 491-525.

4. **Koenker, R., & Xiao, Z. (2006)**. "Quantile Autoregression". *Journal of the American Statistical Association*, 101(475), 980-990.